In [ ]:
import os
import json
import random
import itertools
import warnings
from tqdm.auto import tqdm
from tabulate import tabulate

import torch
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
import lightning as L

import numpy as np
import matplotlib.pyplot as plt

from models import ImageClassificationModel, get_model
from data import MNISTDataset, get_dataset
from utils.plot import *
from xai import *
from metrics import *

plt.rcParams["font.size"] = 16
plt.rcParams["font.family"] = "cmr10"
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
device = "cuda:0"
model_dir = "/home/xia/claim-xai/configs/mnist/base"
# model_dir = "/home/xia/claim-xai/configs/cifar10/base"
# model_dir = "/home/xia/claim-xai/configs/imagenette/base"
# model_dir = "/home/xia/claim-xai/configs/imagenette/ablation/relu"
# model_dir = "/home/xia/claim-xai/configs/imagenette/ablation/sigmoid"
# model_dir = "/home/xia/claim-xai/configs/imagenette/ablation/rescale"
# model_dir = "/home/xia/claim-xai/configs/imagenette/ablation/loss-binary"
# model_dir = "/home/xia/claim-xai/configs/imagenette/ablation/loss-magnitude"
# model_dir = "/home/xia/claim-xai/configs/imagenette/ablation/no-loss"
# model_dir = "/home/xia/claim-xai/configs/imagenette/ablation/nothing"
model_config = json.load( open(os.path.join(model_dir, "config.json"), "r") )
model_metadata = json.load( open(os.path.join(model_dir, "masker/_metadata.json"), "r") )

In [ ]:
ds_train, ds_val, ds_test, num_classes = get_dataset(
    model_config["dataset_name"], 
    data_dir = "../_datasets", 
    splits = ["train", "validation", "test"],
    **model_config["dataset_args"], 
)


model = get_model(
    model_name = model_config["model_name"],
    checkpoint_path = os.path.join(model_dir, model_metadata["best_checkpoint"]),
    **model_config["model_args"]
).eval().to(device)

---

In [ ]:
for i in range(5):
    in_image, label = ds_test[i]

    with torch.no_grad():
        inputs = model.preprocess(in_image.unsqueeze(0))
        logits, weights = model(inputs)

    print(weights.shape)

    image_plot = in_image.cpu()
    weights_plot = weights[0].cpu()
    weights_plot = torchvision.transforms.functional.resize(weights_plot, (image_plot.shape[1], image_plot.shape[2]), interpolation=torchvision.transforms.InterpolationMode.BILINEAR)
    weights_plot = (weights_plot - weights_plot.min()) / (weights_plot.max() - weights_plot.min())


    print(weights_plot.max(), weights_plot.min())
    print(f"pred={torch.argmax(logits).item()} label={label}")


    plot_image_attribution(image_plot, weights_plot)
    plt.show()


# plt.hist(weights_plot.flatten())
# plt.yscale("log")
# plt.show()

---

In [ ]:
in_image, label = ds_test[1]
L.seed_everything(42)

with torch.no_grad():
    inputs = model.preprocess(in_image.unsqueeze(0))
    logits, weights = model(inputs)
pred = torch.argmax(logits).item()
print(f"pred={pred} label={label}")


inputs = (inputs["pixel_values"], )

explainer_our = OurAttribution(model)
attr_our = explainer_our(inputs)[0].mean(dim=1, keepdim=True)

explainer_lig = LayerIntegratedGradientsAttribution(model, baselines=(inputs[0] * 0))
attr_lig = explainer_lig(inputs[0], target=pred)[0].mean(dim=1, keepdim=True)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    explainer_saliency = SaliencyAttribution(model)
    attr_saliency = explainer_saliency(inputs, target=pred)[0].mean(dim=1, keepdim=True)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    explainer_dl = DeepLiftAttribution(model, baselines=(inputs[0] * 0.0))
    attr_dl = explainer_dl(inputs, target=pred)[0].mean(dim=1, keepdim=True)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    baselines_dlshap = torch.cat([ model.preprocess(ds_train[i][0])["pixel_values"] for i in range(20) ], dim=0)
    explainer_dlshap = DeepLiftShapAttribution(model, baselines=baselines_dlshap)
    attr_dlshap = explainer_dlshap(inputs, target=pred)[0].mean(dim=1, keepdim=True)

baselines_grad_shap = torch.cat([ model.preprocess(ds_train[i][0])["pixel_values"] for i in range(20) ], dim=0)
explainer_grad_shap = GradientShapAttribution(model, baselines=baselines_grad_shap)
attr_grad_shap = explainer_grad_shap(inputs[0], target=pred, n_samples=100)[0].mean(dim=1, keepdim=True)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    explainer_gbprop = GuidedBackpropAttribution(model)
    attr_gbprop = explainer_gbprop(inputs, target=pred)[0].mean(dim=1, keepdim=True)

image_plot = in_image.cpu()

for attr in [ attr_our, attr_lig, attr_saliency, attr_dl, attr_dlshap, attr_grad_shap, attr_gbprop ]:
    attr_plot = torchvision.transforms.functional.resize(attr[0].cpu().detach(), (image_plot.shape[1], image_plot.shape[2]), interpolation=torchvision.transforms.InterpolationMode.BILINEAR)
    attr_plot = (attr_plot - attr_plot.min()) / (attr_plot.max() - attr_plot.min())
    plot_image_attribution(image_plot, attr_plot)
    plt.show()

---

In [ ]:
def _accum_metrics(metrics, model_base, inputs, attribution):
    metrics["avg-drop"].accumulate(model_base, inputs, attribution)
    metrics["delete-auc"].accumulate(model_base, inputs, attribution)
    metrics["complexity"].accumulate(attribution)
    metrics["sparsity"].accumulate(attribution)

In [ ]:
L.seed_everything(42)
model_base = get_model_wrapper(model)

methods = [ "ours", "layer-ig", "saliency", "deeplift", "deeplift-shap", "gradient-shap", "guided-backprop" ]
metrics = {
    method: {
        "avg-drop": AverageDropMetric(device),
        "delete-auc": DeletionAUCMetric(device),
        "complexity": ComplexityMetric(device),
        "sparsity": SparsityMetric(device)
    } for method in methods
}


for i in tqdm(range(len(ds_test))):
    in_image, label = ds_test[i]

    with torch.no_grad():
        inputs = model.preprocess(in_image.unsqueeze(0))
        logits, weights = model(inputs)
    pred = torch.argmax(logits).item()


    inputs = (inputs["pixel_values"], )


    explainer_our = OurAttribution(model)
    attr_our = explainer_our(inputs)[0].mean(dim=1, keepdim=True)
    _accum_metrics(metrics["ours"], model_base, inputs, attr_our)

    explainer_lig = LayerIntegratedGradientsAttribution(model, baselines=(inputs[0] * 0))
    attr_lig = explainer_lig(inputs[0], target=pred)[0].mean(dim=1, keepdim=True)
    _accum_metrics(metrics["layer-ig"], model_base, inputs, attr_lig)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        explainer_saliency = SaliencyAttribution(model)
        attr_saliency = explainer_saliency(inputs, target=pred)[0].mean(dim=1, keepdim=True)
        _accum_metrics(metrics["saliency"], model_base, inputs, attr_saliency)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        explainer_dl = DeepLiftAttribution(model, baselines=(inputs[0] * 0.0))
        attr_dl = explainer_dl(inputs, target=pred)[0].mean(dim=1, keepdim=True)
        _accum_metrics(metrics["deeplift"], model_base, inputs, attr_dl)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        baselines_dlshap = torch.cat([ model.preprocess(ds_train[i][0])["pixel_values"] for i in range(20) ], dim=0)
        explainer_dlshap = DeepLiftShapAttribution(model, baselines=baselines_dlshap)
        attr_dlshap = explainer_dlshap(inputs, target=pred)[0].mean(dim=1, keepdim=True)
        _accum_metrics(metrics["deeplift-shap"], model_base, inputs, attr_dlshap)

    baselines_grad_shap = torch.cat([ model.preprocess(ds_train[i][0])["pixel_values"] for i in range(20) ], dim=0)
    explainer_grad_shap = GradientShapAttribution(model, baselines=baselines_grad_shap)
    attr_grad_shap = explainer_grad_shap(inputs[0], target=pred, n_samples=100)[0].mean(dim=1, keepdim=True)
    _accum_metrics(metrics["gradient-shap"], model_base, inputs, attr_grad_shap)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        explainer_gbprop = GuidedBackpropAttribution(model)
        attr_gbprop = explainer_gbprop(inputs, target=pred)[0].mean(dim=1, keepdim=True)
        _accum_metrics(metrics["guided-backprop"], model_base, inputs, attr_gbprop)

    torch.cuda.empty_cache()


In [ ]:
tab_out = []
for method in metrics:
    out = [ method ]
    for metric in metrics[method]:
        mean, std = metrics[method][metric].compute()
        out.append( f"{mean:.2f} +/- {std:.2f}" )
    tab_out.append( out )

print(tabulate(
    tab_out, 
    headers = ["method", *metrics["ours"]]
))

In [ ]:
json_out = {}
for method in metrics:
    json_out[method] = {}
    for metric in metrics[method]:
        mean, std = metrics[method][metric].compute()
        json_out[method][metric] = { "mean": mean, "std": std }

In [ ]:
json.dump(json_out, open(os.path.join(model_dir, "metrics.json"), "w"), indent=4)